In [55]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# 2. Clean imports from your local Python files
from dataset import get_data_loaders
from model import CatConvNet

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import torch

cat_model = CatConvNet()
state_dict = torch.load("/Users/hegedusrazvan/Developer/Faculty/IOTCA/CatPlayground/best_catnet_model.pth", map_location='cpu')
cat_model.load_state_dict(state_dict)
cat_model.to('cpu')
cat_model.eval()

example_inputs = torch.randn(1, 3, 160, 224, device='cpu')

onnx_program = torch.onnx.export(
    cat_model, 
    example_inputs, 
    opset_version=18,        
    input_names=["input"],
    output_names=["heatmap", "size", "offset"]
)
onnx_program.save("cat_centernet.onnx")
print("Export to ONNX complete! Ready for the Raspberry Pi.")

[torch.onnx] Obtain model graph for `CatConvNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CatConvNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Export to ONNX complete! Ready for the Raspberry Pi.


/Users/hegedusrazvan/.local/share/uv/python/cpython-3.14.4-macos-aarch64-none/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [58]:
import onnx

onnx_model = onnx.load("cat_centernet.onnx")
onnx.checker.check_model(onnx_model)

In [59]:
import onnxruntime
onnx_input_array = example_inputs.cpu().numpy()

ort_session = onnxruntime.InferenceSession(
    "./cat_centernet.onnx", providers=["CPUExecutionProvider"]
)

input_name = ort_session.get_inputs()[0].name

onnxruntime_input = {input_name: onnx_input_array}

onnxruntime_outputs = ort_session.run(None, onnxruntime_input)

print("--- ONNX Input Nodes ---")
for idx, input_node in enumerate(ort_session.get_inputs()):
    print(f"Input {idx}: Name = '{input_node.name}', Shape = {input_node.shape}")

print("\n--- ONNX Output Nodes ---")
for idx, output_node in enumerate(ort_session.get_outputs()):
    print(f"Output {idx}: Name = '{output_node.name}', Shape = {output_node.shape}")

--- ONNX Input Nodes ---
Input 0: Name = 'input', Shape = [1, 3, 160, 224]

--- ONNX Output Nodes ---
Output 0: Name = 'heatmap', Shape = [1, 1, 40, 56]
Output 1: Name = 'size', Shape = [1, 2, 40, 56]
Output 2: Name = 'offset', Shape = [1, 2, 40, 56]


In [ ]:
import numpy as np
import torch

cat_model.to('mps')
cat_model.eval()

torch.manual_seed(42)
static_input = torch.randn(1, 3, 160, 224, device='mps')

static_input_numpy = static_input.cpu().numpy()

with torch.no_grad():
    torch_outputs = cat_model(static_input)

ort_outputs = ort_session.run(None, {ort_session.get_inputs()[0].name: static_input_numpy})

print("--- Auto-Discovering ONNX Output Order ---")
final_mapping_indices = {}

for key, pt_tensor in torch_outputs.items():
    match_found = False
    for idx, ort_arr in enumerate(ort_outputs):
        ort_tensor = torch.tensor(ort_arr, device='mps')
        
        if pt_tensor.shape == ort_tensor.shape:
            try:
                torch.testing.assert_close(pt_tensor, ort_tensor, rtol=1e-03, atol=1e-04)
                print(f"PyTorch '{key:<7}' is exactly ONNX output index [{idx}]")
                final_mapping_indices[key] = idx
                match_found = True
                break 
            except AssertionError:
                continue
                
    if not match_found:
        print(f"Could not find a match for '{key}'!")

print("\n--- Copy This for Your Raspberry Pi Script ---")
print("outputs = session.run(None, {input_name: input_tensor})")
print(f"heatmap = outputs[{final_mapping_indices.get('heatmap', '?')}]")
print(f"size    = outputs[{final_mapping_indices.get('size', '?')}]")
print(f"offset  = outputs[{final_mapping_indices.get('offset', '?')}]")

--- Auto-Discovering ONNX Output Order ---
PyTorch 'heatmap' is exactly ONNX output index [0]
PyTorch 'size   ' is exactly ONNX output index [1]
PyTorch 'offset ' is exactly ONNX output index [2]

--- Copy This for Your Raspberry Pi Script ---
outputs = session.run(None, {input_name: input_tensor})
heatmap = outputs[0]
size    = outputs[1]
offset  = outputs[2]
